In [1]:
# Massbezeichnung: Technical Drawing Processor
import os
import numpy as np
from PIL import Image
import gradio as gr
import tkinter as tk
from tkinter import filedialog
import threading
import time
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor
import glob
import itertools
import cv2

def select_folder():
    result = [None]
    def open_dialog():
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        result[0] = filedialog.askdirectory()
        root.destroy()
    thread = threading.Thread(target=open_dialog)
    thread.start()
    thread.join()
    return result[0] if result[0] else ""

def count_processable_images(folder_path):
    return len(glob.glob(os.path.join(folder_path, "*.png"))) + \
           len(glob.glob(os.path.join(folder_path, "*.jpg"))) + \
           len(glob.glob(os.path.join(folder_path, "*.jpeg"))) + \
           len(glob.glob(os.path.join(folder_path, "*.webp")))

def find_drawing_content_bbox(image_path, threshold=245):
    """
    Find bounding box of all non-white content in a technical drawing
    This preserves text, dimension lines, and all drawing elements
    """
    try:
        # Load the image in grayscale
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Try with PIL if OpenCV fails
            pil_img = Image.open(image_path).convert("L")
            img = np.array(pil_img)
        
        # Threshold the image to separate content from white background
        # For technical drawings, we want to keep all non-white pixels
        _, binary = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY_INV)
        
        # Find all non-zero (content) pixels
        non_zero = cv2.findNonZero(binary)
        
        if non_zero is None or len(non_zero) == 0:
            # If no content found, return full image dimensions
            h, w = img.shape[:2]
            return (0, 0, w, h)
        
        # Find the bounding box of all content
        x, y, w, h = cv2.boundingRect(non_zero)
        
        # Add a small margin to ensure no content is cut off
        margin = 10
        x_min = max(0, x - margin)
        y_min = max(0, y - margin)
        x_max = min(img.shape[1], x + w + margin)
        y_max = min(img.shape[0], y + h + margin)
        
        print(f"Drawing content bbox for {image_path}: ({x_min}, {y_min}, {x_max}, {y_max})")
        return (x_min, y_min, x_max, y_max)
    
    except Exception as e:
        print(f"Error finding drawing content for {image_path}: {e}")
        # Return the default full image if an error occurs
        try:
            img = Image.open(image_path)
            return (0, 0, img.width, img.height)
        except:
            return (0, 0, 1000, 1000)  # Fallback default

def process_single_drawing(args):
    """Process a single technical drawing"""
    input_path, output_path, padding, final_size = args
    
    try:
        # Load the original image
        img = Image.open(input_path).convert("RGB")
        
        # Find the bounding box of all drawing content
        bbox = find_drawing_content_bbox(input_path)
        x_min, y_min, x_max, y_max = bbox
        
        # Crop to the content area
        cropped = img.crop((x_min, y_min, x_max, y_max))
        
        # Calculate dimensions for resizing with padding
        content_width = x_max - x_min
        content_height = y_max - y_min
        
        # Apply padding
        available_size = final_size - (2 * padding)
        
        # Use aspect ratio to calculate the scaling factor
        scale_factor = min(
            available_size / content_width,
            available_size / content_height
        )
        
        new_width = int(content_width * scale_factor)
        new_height = int(content_height * scale_factor)
        
        # Resize the cropped content while preserving aspect ratio
        # Use LANCZOS for better quality with text and lines
        resized = cropped.resize((new_width, new_height), Image.LANCZOS)
        
        # Create final image with padding on white background
        output_img = Image.new("RGB", (final_size, final_size), (255, 255, 255))
        
        # Calculate position to center the content
        x_offset = (final_size - new_width) // 2
        y_offset = (final_size - new_height) // 2
        
        # Paste the resized content onto the white background
        output_img.paste(resized, (x_offset, y_offset))
        
        print(f"Processed drawing: {input_path}")
        print(f"Content bbox: {bbox}")
        print(f"Resized to: {new_width}x{new_height}")
        print(f"Final size: {final_size}x{final_size}")
        
        # Save the result
        output_img.save(output_path, format="PNG" if output_path.endswith('.png') else "JPEG", quality=95)
        
        return True
    except Exception as e:
        print(f"Error processing {input_path}: {e}")
        return False

def process_drawings(input_folder, output_folder, padding, final_size, progress=gr.Progress()):
    if not os.path.isdir(input_folder) or not os.path.isdir(output_folder):
        return "Error: Please provide valid folder paths."
    
    # Create output directory
    output_folder_name = f"Output_technical_drawings"
    full_output_path = os.path.join(output_folder, output_folder_name)
    os.makedirs(full_output_path, exist_ok=True)
    
    # Find all images
    image_files = list(itertools.chain(
        glob.glob(os.path.join(input_folder, "*.png")),
        glob.glob(os.path.join(input_folder, "*.jpg")),
        glob.glob(os.path.join(input_folder, "*.jpeg")),
        glob.glob(os.path.join(input_folder, "*.webp"))
    ))
    
    total_images = len(image_files)
    if total_images == 0:
        return "No images found in the input folder."
    
    start_time = time.time()
    progress(0, desc="Starting...")
    
    # Prepare tasks
    tasks = []
    for image in image_files:
        output_path = os.path.join(full_output_path, os.path.splitext(os.path.basename(image))[0] + ".png")
        tasks.append((image, output_path, padding, final_size))
    
    # Process images
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(executor.map(process_single_drawing, tasks))
        
        for i, result in enumerate(results, 1):
            progress(i / total_images, desc=f"Processing {i}/{total_images}")
    
    progress(1.0, desc="Processing complete!")
    total_time = timedelta(seconds=int(time.time() - start_time))
    
    successful = sum(results)
    return f"Processing completed in {total_time}.\n- Processed {successful}/{total_images} technical drawings\n- Output saved to: {full_output_path}"

def create_gradio_interface():
    with gr.Blocks(title="Technical Drawing Processor") as app:
        gr.Markdown("# Technical Drawing Processing Tool")
        gr.Markdown("Automatically processes technical drawings while preserving all text, numbers, dimension lines, and other content. Maintains white background and properly centers the content.")
        
        with gr.Row():
            with gr.Column():
                input_folder = gr.Textbox(label="Input Folder Path")
                input_btn = gr.Button("Select Input Folder")
                output_folder = gr.Textbox(label="Output Folder Path")
                output_btn = gr.Button("Select Output Folder")
                
                with gr.Row():
                    padding = gr.Slider(minimum=0, maximum=200, value=80, step=10, label="Padding (px)")
                    final_size = gr.Slider(minimum=500, maximum=5000, value=1500, step=100, label="Final Image Size (px)")
                
                process_btn = gr.Button("Process Drawings", variant="primary")
            
            with gr.Column():
                output = gr.Textbox(label="Processing Results", lines=10)
        
        input_btn.click(fn=select_folder, outputs=input_folder)
        output_btn.click(fn=select_folder, outputs=output_folder)
        
        process_btn.click(
            fn=process_drawings,
            inputs=[input_folder, output_folder, padding, final_size],
            outputs=output
        )
    
    return app

if __name__ == "__main__":
    app = create_gradio_interface()
    app.launch(share=True)

c:\Users\r.musawi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://271c388dfe6eb5ac63.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\1287102-Extraflame-Evolution-Line-ISIDORA-IDRO-H20-50-BORD-wassergefuehrter-Pelletofen-mz.png: (0, 0, 709, 798)Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\3001240-La-Nordica-Giulietta-16-Kaminofen-mz.png: (95, 24, 689, 762)
Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\3001101-La-Nordica-Gemma-Forno-16-Kaminofen-Elegance-Weiss-Infinity-mz.png: (115, 0, 578, 773)
Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\1287101-Extraflame-Evolution-Line-ISIDORA-IDRO-H20-50-BIAN-wassergefuehrter-Pelletofen-mz.png: (0, 0, 709, 798)
Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\3001110-La-Nordica-Gemma-Forno-16-Kaminofen-Naturstein-mz.png: (115, 0, 578, 773)
Drawing content bbox for C:\Users\r.musawi\Documents\lanordica-28-08-2025\img\mz\1287100-Extraflame-Evolution-Line-ISIDORA-IDR